In [2]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")
from utils.scaling import fit_scalers, apply_scaler

from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aeema_articulatory_features

In [3]:
df = pd.read_csv("../dataset_construction/datasets/processed_aeema.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])

In [4]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,1.203764,1.273764,1.273764,1.353764,m,æ,m,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
1,1.273764,1.353764,1.353764,1.443764,æ,s,m_æ,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
2,1.353764,1.443764,1.443764,1.513764,s,k,m_æ_s,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
3,1.443764,1.513764,1.513764,1.653764,k,ɝ,m_æ_s_k,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
4,1.513764,1.653764,1.653764,1.763764,ɝ,eɪ,m_æ_s_k_ɝ,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
...,...,...,...,...,...,...,...,...,...,...,...,...
43454,23.919070,23.999070,23.999070,24.089070,ʃ,ʌ,k_ɑ_m_p_ʌ_t_ɪ_ʃ,competition,35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
43455,23.999070,24.089070,24.089070,24.149070,ʌ,n,k_ɑ_m_p_ʌ_t_ɪ_ʃ_ʌ,competition,35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
43456,24.149070,24.179070,24.179070,24.209070,h,w,h,which,36.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
43457,24.179070,24.209070,24.209070,24.239070,w,ɪ,h_w,which,36.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import SequenceAutoencoder

device = "mps" if torch.backends.mps.is_available() else "cpu"

art_model = SequenceAutoencoder(
    feature_shape=(6, 3),
    latent_dim=64,
).to(device)
ckpt = torch.load(
    "../../encoder_training/ckpts/aeema/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/aeema/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [6]:
with open(f"../../encoder_training/scalers/aeema_art.pkl", "rb") as f:
    art_scalers = pickle.load(f)
with open(f"../../encoder_training/scalers/aeema_ac.pkl", "rb") as f:
    ac_scalers = pickle.load(f)

In [7]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,1.203764,1.273764,1.273764,1.353764,m,æ,m,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
1,1.273764,1.353764,1.353764,1.443764,æ,s,m_æ,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
2,1.353764,1.443764,1.443764,1.513764,s,k,m_æ_s,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
3,1.443764,1.513764,1.513764,1.653764,k,ɝ,m_æ_s_k,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
4,1.513764,1.653764,1.653764,1.763764,ɝ,eɪ,m_æ_s_k_ɝ,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
...,...,...,...,...,...,...,...,...,...,...,...,...
43454,23.919070,23.999070,23.999070,24.089070,ʃ,ʌ,k_ɑ_m_p_ʌ_t_ɪ_ʃ,competition,35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
43455,23.999070,24.089070,24.089070,24.149070,ʌ,n,k_ɑ_m_p_ʌ_t_ɪ_ʃ_ʌ,competition,35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
43456,24.149070,24.179070,24.179070,24.209070,h,w,h,which,36.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
43457,24.179070,24.209070,24.209070,24.239070,w,ɪ,h_w,which,36.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1


In [8]:
def get_vectors(art_f, ac_f, start, end, subject):

    articulatory_features = get_aeema_articulatory_features(art_f, start, end)
    if articulatory_features is not None:
        articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 6, 3)
        articulatory_features = apply_scaler(articulatory_features, art_scalers[subject])
        articulatory_features = torch.from_numpy(articulatory_features).float().unsqueeze(0).to(device)
        art_out, art_z = art_model(articulatory_features, return_latent=True)
    else:
        art_out, art_z = None, None

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is not None:
        mfccs = apply_scaler(mfccs, ac_scalers[subject])
        mfccs = torch.from_numpy(mfccs).unsqueeze(0).to(device)
        ac_out, ac_z = ac_model(mfccs, return_latent=True)
    else:
        ac_out, ac_z = None, None

    return {
        "articulatory_features": (
            articulatory_features.detach().cpu().numpy()
            if articulatory_features is not None
            else None
        ),
        "art_z": (
            art_z.detach().cpu().numpy()
            if art_z is not None
            else None
        ),
        "mfccs": (
            mfccs.detach().cpu().numpy()
            if mfccs is not None
            else None
        ),
        "ac_z": (
            ac_z.detach().cpu().numpy()
            if ac_z is not None
            else None
        ),
    }

In [9]:
def distance(v1, v2):
    if v1 is None or v2 is None:
        return None
    return np.linalg.norm(v1 - v2)

In [10]:
pbar = tqdm.tqdm(df.iterrows(), total=len(df))

dists = {
    "art_latent_dist": [],
    # "art_raw_dist": [],
    "ac_latent_dist": [],
    # "ac_raw_dist": [],
}
for idx, data in pbar:
    
    phone1_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone1_start"], data["phone1_end"], data["subject"])
    phone2_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone2_start"], data["phone2_end"], data["subject"])

    art_latent_dist = distance(phone1_vectors["art_z"], phone2_vectors["art_z"])
    # art_raw_dist = distance(phone1_vectors["articulatory_features"], phone2_vectors["articulatory_features"])

    ac_latent_dist = distance(phone1_vectors["ac_z"], phone2_vectors["ac_z"])
    # ac_raw_dist = distance(phone1_vectors["mfccs"], phone2_vectors["mfccs"])

    dists["art_latent_dist"].append(art_latent_dist)
    # dists["art_raw_dist"].append(art_raw_dist)
    dists["ac_latent_dist"].append(ac_latent_dist)
    # dists["ac_raw_dist"].append(ac_raw_dist)

pd.DataFrame(dists).to_csv("distances_aeema.csv", index=False)

  0%|          | 0/43459 [00:00<?, ?it/s]

100%|██████████| 43459/43459 [1:43:01<00:00,  7.03it/s]  
